# Simulación reproducible de manos independientes de 5 cartas

Cada mano se extrae de un mazo estándar completo. Las manos son independientes entre sí: el mazo se vuelve a considerar completo antes de generar la siguiente mano.

Ejecutá el notebook completo desde esta celda para reproducir el experimento.

In [ ]:
from collections import Counter
from random import Random

SEMILLA = 20260805
NUMERO_DE_MANOS = 100_000

generador = Random(SEMILLA)

## Mazo y generación

Una carta se representa como una tupla `(valor, palo)`. `sample` elige cinco cartas sin repetición dentro de cada mano.

In [ ]:
VALORES = tuple(range(2, 15))  # 11=J, 12=Q, 13=K, 14=A
PALOS = ("♠", "♥", "♦", "♣")
MAZO = tuple((valor, palo) for valor in VALORES for palo in PALOS)

def generar_mano(generador):
    """Devuelve una mano nueva e independiente de cinco cartas."""
    return tuple(generador.sample(MAZO, k=5))

def mostrar_carta(carta):
    nombres = {11: "J", 12: "Q", 13: "K", 14: "A"}
    valor, palo = carta
    return f"{nombres.get(valor, valor)}{palo}"

def mostrar_mano(mano):
    return " ".join(mostrar_carta(carta) for carta in mano)

## Clasificación

La clasificación usa las cinco cartas de la mano. Una escalera puede terminar en as o usar el as como carta baja (A-2-3-4-5).

In [ ]:
def es_escalera(valores):
    valores_unicos = sorted(set(valores))
    if len(valores_unicos) != 5:
        return False
    if valores_unicos == [2, 3, 4, 5, 14]:
        return True
    return valores_unicos[-1] - valores_unicos[0] == 4

def clasificar_mano(mano):
    valores = [valor for valor, _ in mano]
    palos = [palo for _, palo in mano]
    repeticiones = sorted(Counter(valores).values(), reverse=True)
    color = len(set(palos)) == 1
    escalera = es_escalera(valores)

    if color and set(valores) == {10, 11, 12, 13, 14}:
        return "Escalera real"
    if color and escalera:
        return "Escalera de color"
    if repeticiones == [4, 1]:
        return "Póker"
    if repeticiones == [3, 2]:
        return "Full house"
    if color:
        return "Color"
    if escalera:
        return "Escalera"
    if repeticiones == [3, 1, 1]:
        return "Trío"
    if repeticiones == [2, 2, 1]:
        return "Doble pareja"
    if repeticiones == [2, 1, 1, 1]:
        return "Pareja"
    return "Carta alta"

## Simulación y resumen

Al volver a crear el generador con la misma semilla, esta celda entrega exactamente la misma muestra aunque se ejecute varias veces.

In [ ]:
generador = Random(SEMILLA)
manos = [generar_mano(generador) for _ in range(NUMERO_DE_MANOS)]
conteos = Counter(clasificar_mano(mano) for mano in manos)

orden = (
    "Escalera real", "Escalera de color", "Póker", "Full house",
    "Color", "Escalera", "Trío", "Doble pareja", "Pareja", "Carta alta",
)

print(f"Semilla: {SEMILLA} | Manos simuladas: {NUMERO_DE_MANOS:,}")
print(f"Ejemplo de mano: {mostrar_mano(manos[0])} ({clasificar_mano(manos[0])})\n")
for categoria in orden:
    cantidad = conteos[categoria]
    print(f"{categoria:20} {cantidad:>7,}  {cantidad / NUMERO_DE_MANOS:>8.4%}")

### Comprobaciones rápidas

Estas condiciones verifican que todas las manos tienen cinco cartas distintas y que se clasificó cada observación.

In [ ]:
assert len(MAZO) == 52
assert all(len(mano) == 5 and len(set(mano)) == 5 for mano in manos)
assert sum(conteos.values()) == NUMERO_DE_MANOS

print("Verificaciones superadas.")